# LAr optical map from simulation counts

Learn one detection-probability map from simulations that share the same physics
but use different numbers of primaries. LF/HF are source and split labels only.
The GP input is voxel position; the training observations are integer counts
of hit-producing primaries **m**, with their actual simulated primary counts **N**.

$$f(\theta)\sim GP,\quad p(\theta)=\operatorname{sigmoid}(f(\theta)),\quad
m_i\sim\operatorname{Binomial}(N_i,p(\theta_i)).$$

This notebook uses a separate binomial-likelihood GP backend with Laplace
approximate inference. There is no CNP, mixup, context split, pseudo-count target
or separate fidelity function. Each file’s full primary count is used. Kernel parameters
are optimized on training data only; posterior uncertainty is conditional on them.

The HF configuration includes both zero-hit and nonzero-hit files. The data-count
table below reports the selected folders’ contents and split. The likelihood is
an ordinary, untruncated binomial likelihood and handles zero counts directly.
Selecting only nonzero files for any source can bias population inference.

Use the repository `.venv` with the `gp` and `optical-data` dependencies installed.


In [ ]:
import os
for name in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ[name] = "1"

import sys
import json
from pathlib import Path
from datetime import datetime, timezone
from IPython.display import Image, Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "config.optical.binomial.yaml").exists())
sys.path.insert(0, str(ROOT))
from schemas.count_gp import load_count_gp_config
from data.optical_counts import prepare_optical_counts
from core.count_gp_experiment import run_count_gp

## Configuration

All inputs, split policy and GP settings are in `config.optical.binomial.yaml`.
This config has its own prepared-data directory, split manifest and result folder.
It does not modify the CNP/MFGP experiment. Defaults use all selected LF voxels and
10 HF voxels for training; remaining HF voxels are validation. Same-center split
conflicts are rejected. Input scaling is fitted using training coordinates only.

`RUN_TRAINING = True` fits a fresh GP synchronously. Set it to `False` to inspect
an existing run without reading the raw simulation files or refitting.


In [ ]:
RUN_TRAINING = True
EXISTING_RUN = None  # Optional path; otherwise use the latest count-GP run.

config = load_count_gp_config(ROOT / "config.optical.binomial.yaml")
if RUN_TRAINING:
    run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    RUN_DIR = config.output_directory / run_id
    config = config.model_copy(update={"output_directory": RUN_DIR})
else:
    saved = sorted(config.output_directory.glob("*/metrics.json"))
    if EXISTING_RUN is None and not saved:
        raise FileNotFoundError("No completed count-GP run. Set RUN_TRAINING = True.")
    RUN_DIR = Path(EXISTING_RUN).expanduser() if EXISTING_RUN else saved[-1].parent
    if not RUN_DIR.is_absolute():
        RUN_DIR = ROOT / RUN_DIR
print("Run directory:", RUN_DIR)

## Prepare full-run counts

Each primary contributes one binary outcome: whether it produced any hit.
Multiple optical hits from one primary count once. Folder changes automatically
update this workflow's count table and split manifest. Different event counts
within the same source group are supported. No data are padded or resampled.


In [ ]:
prepared = prepare_optical_counts(config.data) if RUN_TRAINING else None
rows = prepared[1] if prepared is not None else json.loads((RUN_DIR / "counts.json").read_text())
text = "| Split | Source | Voxels | Primaries per voxel | Total hits | Zero-hit voxels |\n"
text += "|---|---|---:|---|---:|---:|\n"
for split, group in sorted({(r["split"], r["source_group"]) for r in rows}):
    selected = [r for r in rows if r["split"] == split and r["source_group"] == group]
    budgets = ", ".join(map(str, sorted({r["trials"] for r in selected})))
    text += f"| {split} | {group} | {len(selected)} | {budgets} | {sum(r['hits'] for r in selected)} | {sum(r['hits'] == 0 for r in selected)} |\n"
display(Markdown(text))

## Fit the shared probability map

The GP uses a spatial Matérn-5/2 kernel plus a constant kernel for the global
log-odds. A logistic link keeps probabilities in (0,1). The likelihood uses each
row's own N directly. No validation outcomes enter fitting or model selection.
Check the saved optimizer status: a finite iteration limit need not imply convergence.


In [ ]:
if RUN_TRAINING:
    RUN_DIR = run_count_gp(config, prepared)
print("Results:", RUN_DIR)
display(json.loads((RUN_DIR / "model.json").read_text()))

## Validation at exact voxel coordinates

The mean is the posterior expected detection probability. Predictive intervals
sample the latent posterior, apply the logistic link, and generate binomial
counts using the validation voxel's full N. They include counting noise and
approximate GP uncertainty, without an extra Gaussian noise term or spatial pooling.

The 68.27%, 95.45% and 99.73% intervals are discrete Monte Carlo quantiles;
“σ probability” labels describe nominal probability mass, not symmetric bands.
Coverage describes the configured held-out sample; spatial representativeness
and any outcome-based selection still matter for population interpretation.


In [ ]:
metrics = json.loads((RUN_DIR / "metrics.json").read_text())
display(metrics)
for row in metrics:
    for plot in ("means", "coverage"):
        display(Image(filename=str(RUN_DIR / f"{row['source_group']}_{plot}.png")))

## Spatial projections

Both GP approaches use the same physical-volume projection engine. The adapters
supply joint latent response draws; predictive projections add binomial counting
noise separately. `latent_mean` shows uncertainty in a spatial average;
`observed_fraction` retains spatial variation and describes individual simulated
voxels. Individual observations are not coverage measurements for averaged bands.

The settings are in `projections` in the YAML. These cells can also project a
saved run with `RUN_TRAINING=False`, without retraining. With multiple validation
budgets, set `target_events` explicitly. For comparisons between models, specify
the same physical box, grid, and target-event budget.

In [ ]:
from core.count_gp_projections import ensure_count_gp_projections

if config.projections.enabled:
    for quantity in ("latent_mean", "observed_fraction"):
        settings = config.projections.model_copy(update={"quantity": quantity})
        projection_dir = ensure_count_gp_projections(RUN_DIR, settings)
        title = "Marginalized response — uncertainty in the spatial average" if quantity == "latent_mean" else "Individual-voxel prediction — spatial variation and counting noise"
        display(Markdown(f"### {title}"))
        print("Saved plots:", projection_dir)
        # Mean bands are shown without individual-voxel overlays.
        names = ("curves", "planes") if quantity == "latent_mean" else ("curves_observed", "planes_observed")
        for name in names:
            display(Image(filename=str(projection_dir / f"{name}.png")))